In [1]:
!pip install pandas numpy scikit-learn torch transformers datasets sentence-transformers faiss-cpu matplotlib seaborn

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sentence_transformers import SentenceTransformer
import faiss
import pickle

# ---------------------------
# 1. LOAD DATA
# ---------------------------

df1 = pd.read_csv("Combined Data.csv")
df2 = pd.read_csv("Mental_Health_FAQ.csv")
df3 = pd.read_csv("Suicide_Detection.csv")

# Standardize
df3 = df3.rename(columns={"class": "label"})
df2["text"] = df2["Questions"]
df2["label"] = "faq"

df = pd.concat([
    df1[["text", "label"]],
    df2[["text", "label"]],
    df3[["text", "label"]]
], ignore_index=True)

df.dropna(inplace=True)

# ---------------------------
# 2. LABEL MAPPING
# ---------------------------

def map_label(label):
    label = str(label).lower()

    if "suicide" in label:
        return "suicide"
    elif "depress" in label or "sad" in label:
        return "depression"
    elif "anxiety" in label or "stress" in label:
        return "anxiety"
    elif "faq" in label:
        return "faq"
    else:
        return "normal"

df["label"] = df["label"].apply(map_label)

label2id = {l: i for i, l in enumerate(df["label"].unique())}
id2label = {i: l for l, i in label2id.items()}
df["label"] = df["label"].map(label2id)

# ---------------------------
# 3. BALANCE DATA
# ---------------------------

min_count = df["label"].value_counts().min()

df_balanced = df.groupby("label").apply(
    lambda x: x.sample(min_count)
).reset_index(drop=True)

# ---------------------------
# 4. TRAIN TEST SPLIT
# ---------------------------

train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.2,
    stratify=df_balanced["label"],
    random_state=42
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# ---------------------------
# 5. TOKENIZATION
# ---------------------------

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# ---------------------------
# 6. MODEL
# ---------------------------

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id)
)

# ---------------------------
# 7. METRICS
# ---------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# ---------------------------
# 8. TRAINING CONFIG
# ---------------------------

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True
)

# ---------------------------
# 9. TRAIN BERT
# ---------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# ---------------------------
# 10. EVALUATION
# ---------------------------

preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

print("\n🔥 Accuracy:", accuracy_score(labels, preds))
print("\n📊 Report:\n")
print(classification_report(labels, preds, target_names=label2id.keys()))

# ---------------------------
# 11. CONFUSION MATRIX
# ---------------------------

cm = confusion_matrix(labels, preds)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

# ---------------------------
# 12. SAVE BERT MODEL
# ---------------------------

model.save_pretrained("models/bert_classifier")
tokenizer.save_pretrained("models/bert_classifier")

with open("models/label_map.pkl", "wb") as f:
    pickle.dump(label2id, f)

# =========================================================
# 🔥 PART 2: RETRIEVAL SYSTEM (FAISS + EMBEDDINGS)
# =========================================================

print("\n🚀 Building Retrieval System...")

faq_df = pd.read_csv("Mental_Health_FAQ.csv")

questions = faq_df["Questions"].tolist()
answers = faq_df["Answers"].tolist()

# Sentence Transformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(questions)

# FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Save everything
faiss.write_index(index, "models/faiss_index.bin")

with open("models/faq_data.pkl", "wb") as f:
    pickle.dump({"questions": questions, "answers": answers}, f)

print("✅ Training Complete. Models Saved.")